# Task-vector negation: algorithm and training

Run the cells in order. The CPU example calls the same objective implementation as the production trainer. The full training cell runs the actual algorithm with an approved frozen recipe, reviewed streams and a staged checkpoint. Results are optional.

Acquisition uses positive completion CE. Negation preserves acquired A and scales B by a negative development-selected factor. Scale zero is a valid null intervention. The production trainer evaluates the frozen development scale grid under its declared retention gate.

## Environment

Use `requirements-unlearning.txt` in an isolated environment. The CPU example needs PyTorch. Full training uses the qualified CUDA build and the pinned model/data recipe. Repository code lives in `src/fairforget/unlearning/`; it is not downloaded by the notebook.

In [ ]:
import json
import os
import sys
from pathlib import Path

import torch

repo = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "src/fairforget/unlearning/objectives.py").is_file()
    ),
    None,
)
if repo is None:
    raise FileNotFoundError(
        "Open this notebook from a checked-out fair_forget repository containing src/fairforget/unlearning."
    )
sys.path.insert(0, str(repo / "src"))
from fairforget.unlearning import objectives, rmu, trainer  # noqa: E402, F401

print("Algorithm source:", repo / "src/fairforget/unlearning")
print("PyTorch:", torch.__version__)

METHOD = "task_vector"
RUN_TRAINING = False

## Executable algorithm step

The following synthetic CPU example checks the implementation. Its tensors and learning rate are software examples, not study settings or measured model results.

In [ ]:
torch.manual_seed(42)
# One tiny linear-LoRA layer: A is acquired, B starts at zero.
initial = {"layer.lora_A.weight": torch.randn(2, 5), "layer.lora_B.weight": torch.zeros(5, 2)}
a = torch.nn.Parameter(initial["layer.lora_A.weight"].clone())
b = torch.nn.Parameter(initial["layer.lora_B.weight"].clone())
inputs = torch.randn(2, 3, 5)
base_logits = torch.randn(2, 3, 5)
labels = torch.tensor([[0, 1, 2], [1, 2, 3]])
mask = torch.tensor([[False, True, True], [False, True, True]])
optimizer = torch.optim.AdamW([a, b], lr=0.01)
optimizer.zero_grad(set_to_none=True)
logits = base_logits + inputs @ (b @ a).T
logps = logits.log_softmax(-1).gather(-1, labels.unsqueeze(-1)).squeeze(-1)
acquisition_loss = objectives.completion_ce_loss(logps, mask, reduction="example_mean")
acquisition_loss.backward()
optimizer.step()
positive = {"layer.lora_A.weight": a.detach().clone(), "layer.lora_B.weight": b.detach().clone()}
# A software-example scale; the full trainer selects its scale on development data.
scale = 0.5
negated = objectives.negate_lora_state(positive, scale=scale, initial_state=initial)
torch.testing.assert_close(
    negated["layer.lora_B.weight"] @ negated["layer.lora_A.weight"],
    -scale * (positive["layer.lora_B.weight"] @ positive["layer.lora_A.weight"]),
)
print("Positive acquisition and B-only task-vector negation completed on CPU.")

## Approved full-model training

Set `FAIRFORGET_RELEASE` to the frozen release directory, `FAIRFORGET_RECIPE` to its absolute recipe file in the frozen release and `FAIRFORGET_OUTPUT` to a new result directory. The recipe must retain its bound reviewed data, source hashes, model revision, masks and development split. Validation is local. Training starts only when you set `RUN_TRAINING=True`.

This notebook uses the **WarBias** protocol. See the [deployment guide](../../deploy/unlearning_primary/README.md) for qualified releases and cluster execution. Model weights and reviewed raw streams remain external to GitHub.

In [ ]:
# Select one approved frozen recipe; keep its reviewed data and model hashes.
release_value = os.environ.get("FAIRFORGET_RELEASE")
RELEASE = Path(release_value).expanduser().resolve() if release_value else None
recipe_value = os.environ.get("FAIRFORGET_RECIPE")
RECIPE = Path(recipe_value).expanduser().resolve() if recipe_value else None
output_value = os.environ.get("FAIRFORGET_OUTPUT")
OUTPUT = Path(output_value).expanduser().resolve() if output_value else None
validated = None
if RECIPE is not None:
    validated, streams, provenance = trainer.validate_config(RECIPE, methods={METHOD})
    print("Validated method:", validated["method"])
    print("Model:", validated["model_id"], validated["model_revision"])
    print("Reviewed stream rows:", {role: len(rows) for role, rows in streams.items()})
else:
    print("For full-model training, set FAIRFORGET_RECIPE to an approved frozen recipe.")

In [ ]:
if RUN_TRAINING:
    if validated is None or OUTPUT is None or RELEASE is None:
        raise ValueError(
            "Set FAIRFORGET_RELEASE, FAIRFORGET_RECIPE and FAIRFORGET_OUTPUT, then validate the recipe."
        )
    if not RECIPE.is_relative_to(RELEASE):
        raise ValueError("The selected recipe must belong to the frozen release.")
    from importlib.metadata import version

    required_versions = {
        "torch": "2.9.1",
        "transformers": "4.57.6",
        "peft": "0.17.1",
        "accelerate": "1.15.0",
    }
    for package, expected in required_versions.items():
        if version(package).split("+")[0] != expected:
            raise RuntimeError(
                f"Use requirements-unlearning.txt: {package}=={expected} is required for full training."
            )
    if not torch.cuda.is_available():
        raise RuntimeError("Full-model training requires the qualified CUDA runtime.")
    if OUTPUT == RELEASE or OUTPUT.is_relative_to(RELEASE):
        raise ValueError("Use a new output directory outside the frozen recipe inputs.")
    if METHOD == "rmu":
        receipt = rmu.run(RECIPE, OUTPUT)
    else:
        receipt = trainer.run(RECIPE, OUTPUT)
    print(json.dumps(receipt, ensure_ascii=False, indent=2))
else:
    print(
        "CPU algorithm example completed. Full-model training is disabled until RUN_TRAINING=True."
    )

## Implementation and checks

- [Loss implementations](../../src/fairforget/unlearning/objectives.py)
- [Full NPO, WGA, task-vector and BiasUnlearn trainer](../../src/fairforget/unlearning/trainer.py)
- [Full fixed-target RMU trainer](../../src/fairforget/unlearning/rmu.py)
- [Objective, masking and trainer checks](../../tests/)

The trainer writes the checkpoint, numeric history, actual valid-token exposure and provenance to the new output directory. Evaluation and reporting are separate workflows; use the [notebook index](../README.md) to find saved reports.